In [3]:
!pip install optuna
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 15.1 MB/s eta 0:00:00


In [4]:
import os
import joblib
import numpy as np
import pandas as pd
import optuna
from google.colab import drive
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss

# Evitar alertas excesivas de Optuna por consola
optuna.logging.set_verbosity(optuna.logging.WARNING)

# =====================================================================
# 0. CONFIGURACIÓN Y MONTAJE DE GOOGLE DRIVE
# =====================================================================
print("🔌 Conectando con Google Drive...")
drive.mount('/content/drive')

ruta_drive = '/content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA'
os.makedirs(ruta_drive, exist_ok=True)
ruta_csv = os.path.join(ruta_drive, 'datos_churn_bancario.csv')

# =====================================================================
# 1. GENERACIÓN DE DATASET DE FUGA DE CLIENTES (CHURN)
# =====================================================================
print("\n🏃 Creando dataset sintético de Churn Bancario...")
np.random.seed(55)
n_usuarios = 2500

# Simulación de variables transaccionales bancarias
antiguedad_meses = np.random.randint(6, 120, size=n_usuarios)
score_satisfaccion = np.random.randint(1, 6, size=n_usuarios) # 1 a 5 estrellas
productos_activos = np.random.randint(1, 5, size=n_usuarios)
balance_cuenta = np.clip(np.random.normal(5000, 4000, n_usuarios), 0, None)
transacciones_mes = np.random.poisson(lam=12, size=n_usuarios)

# Regla de fuga latente: baja satisfacción y pocas transacciones disparan el churn
log_odds = 2.0 - (score_satisfaccion * 1.2) - (productos_activos * 0.5) - (transacciones_mes * 0.08) + (balance_cuenta * 0.0001)
probabilidad_fuga = 1 / (1 + np.exp(-log_odds))
target_churn = np.where(probabilidad_fuga > np.random.uniform(0, 1, n_usuarios), 1, 0)

df_churn = pd.DataFrame({
    'Antiguedad_Meses': antiguedad_meses,
    'Score_Satisfaccion': score_satisfaccion,
    'Productos_Activos': productos_activos,
    'Balance_Cuenta': balance_cuenta,
    'Transacciones_Mes': transacciones_mes,
    'Target_Churn': target_churn
})
df_churn.to_csv(ruta_csv, index=False)
print(f"✅ Dataset de Churn guardado exitosamente en: {ruta_csv}")

# =====================================================================
# 2. PREPARACIÓN DE DATOS Y OPTIMIZACIÓN BAYESIANA (OPTUNA)
# =====================================================================
features = ['Antiguedad_Meses', 'Score_Satisfaccion', 'Productos_Activos', 'Balance_Cuenta', 'Transacciones_Mes']
X = df_churn[features]
y = df_churn['Target_Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("🧠 Iniciando estudio de Optuna para buscar hiperparámetros óptimos...")
def objetivo(trial):
    # Espacio de búsqueda Bayesiano definido por el Data Scientist
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 250),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42,
        'eval_metric': 'logloss'
    }

    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    preds = model.predict_proba(X_test)
    return log_loss(y_test, preds)

estudio = optuna.create_study(direction='minimize')
estudio.optimize(objetivo, n_trials=20) # 20 iteraciones inteligentes

print(f"🥇 Mejores parámetros encontrados: {estudio.best_params}")

# =====================================================================
# 3. ENTRENAMIENTO FINAL Y SERIALIZACIÓN PERSISTENTE (.PKL)
# =====================================================================
print("\n🏋️ Entrenando modelo definitivo de XGBoost con la configuración ganadora...")
modelo_final = XGBClassifier(**estudio.best_params, random_state=42, eval_metric='logloss')
modelo_final.fit(X_train, y_train)

# Estructura para exportación analítica
artefactos_churn = {
    'modelo_xgb': modelo_final,
    'X_test': X_test,
    'y_test': y_test,
    'features': features
}

ruta_pkl_datos = os.path.join(ruta_drive, 'modelo_churn_bancario.pkl')
joblib.dump(artefactos_churn, ruta_pkl_datos)

print(f" -> Modelo final y matrices guardados exitosamente en: {ruta_pkl_datos}")
print("\n🎉 ¡BLOQUE 1 COMPLETADO! Pasa al Bloque 2 para las visualizaciones avanzadas.")


🔌 Conectando con Google Drive...
Mounted at /content/drive

🏃 Creando dataset sintético de Churn Bancario...
✅ Dataset de Churn guardado exitosamente en: /content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA/datos_churn_bancario.csv
🧠 Iniciando estudio de Optuna para buscar hiperparámetros óptimos...
🥇 Mejores parámetros encontrados: {'n_estimators': 83, 'max_depth': 3, 'learning_rate': 0.09377707477634871, 'subsample': 0.9325220277601455, 'colsample_bytree': 0.8043484563477605}

🏋️ Entrenando modelo definitivo de XGBoost con la configuración ganadora...
 -> Modelo final y matrices guardados exitosamente en: /content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA/modelo_churn_bancario.pkl

🎉 ¡BLOQUE 1 COMPLETADO! Pasa al Bloque 2 para las visualizaciones avanzadas.


In [5]:
import os
import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from google.colab import drive
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, roc_auc_score, confusion_matrix, roc_curve

# =====================================================================
# 0. CARGA DE ARTEFACTOS DESDE DRIVE
# =====================================================================
print("🔌 Conectando con Google Drive e importando binarios de Churn...")
drive.mount('/content/drive')

ruta_drive = '/content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA'
ruta_pkl_datos = os.path.join(ruta_drive, 'modelo_churn_bancario.pkl')

if os.path.exists(ruta_pkl_datos):
    artefactos = joblib.load(ruta_pkl_datos)
    modelo_xgb = artefactos['modelo_xgb']
    X_test = artefactos['X_test']
    y_test = artefactos['y_test']
    features = artefactos['features']
    print("✅ Modelo final de XGBoost cargado correctamente.")
else:
    print("❌ Error: No se encontró el archivo .pkl. Ejecuta el Bloque 1.")

# Generación de inferencias predictivas
predicciones_prob = modelo_xgb.predict_proba(X_test)[:, 1]
predicciones_binarias = modelo_xgb.predict(X_test)

# =====================================================================
# 1. GRÁFICA 1: IMPORTANCIA DE VARIABLES (FORMATO .4f% Y EXPLICACIÓN OUTSIDE)
# =====================================================================
print("\n📊 Generando Gráfica 1: Importancia de Variables en Churn...")
importancias = modelo_xgb.feature_importances_
valores_porcentaje = (importancias / np.sum(importancias)) * 100
indices_imp = np.argsort(valores_porcentaje)

df_imp = pd.DataFrame({
    'Variable': [features[i] for i in indices_imp],
    'Importancia': valores_porcentaje[indices_imp]
})
etiquetas_fijas = [f"{v:.4f}%" for v in df_imp['Importancia']]

fig1 = px.bar(
    df_imp, x='Importancia', y='Variable', orientation='h',
    title='🎯 Gráfica 1: Importancia de Características del Cliente en la Retención Bancaria',
    color='Importancia', color_continuous_scale='Mint', text=etiquetas_fijas
)
fig1.update_traces(
    textposition='outside',
    hovertemplate='Variable: %{y}<br>Importancia Relativa: %{x:.4f}%<extra></extra>'
)
fig1.update_layout(template='plotly_white', coloraxis_showscale=False, xaxis=dict(ticksuffix="%", range=[0, max(df_imp['Importancia']) * 1.20]))
fig1.show()

# =====================================================================
# 2. NUEVA GRÁFICA 2: MATRIZ DE CONFUSIÓN INTERACTIVA (MÉTRICA DE CONTROL)
# =====================================================================
print("📊 Generando Gráfica 2: Matriz de Confusión...")
matriz_conf = confusion_matrix(y_test, predicciones_binarias)

# Formatear el mapa de calor con texto estructurado para el sector financiero
z_text = [[str(val) for val in fila] for fila in matriz_conf]
fig2 = ff_fig = px.imshow(
    matriz_conf,
    x=['No Fuga (Permanecer)', 'Fuga (Churn)'],
    y=['No Fuga (Permanecer)', 'Fuga (Churn)'],
    labels=dict(x="Predicción del Modelo", y="Realidad del Cliente", color="Clientes"),
    color_continuous_scale='Blues',
    title='📊 Gráfica 2: Matriz de Confusión para Evaluación de Mitigación de Fuga'
)
fig2.update_traces(
    text=z_text, texttemplate="%{text}",
    hovertemplate='Predicho: %{x}<br>Real: %{y}<br>Cantidad de Clientes: %{z}<extra></extra>'
)
fig2.update_layout(template='plotly_white')
fig2.show()

# =====================================================================
# 3. NUEVA GRÁFICA 3: CURVA ROC INTERACTIVA (FORMATO HOVER .4f)
# =====================================================================
print("📊 Generando Gráfica 3: Curva ROC...")
fpr, tpr, thresholds = roc_curve(y_test, predicciones_prob)

fig3 = go.Figure()
# Línea base de azar
fig3.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Línea de Azar (0.5000)', line=dict(color='red', dash='dash')))
# Curva ROC matemática del modelo
fig3.add_trace(go.Scatter(
    x=fpr, y=tpr, mode='lines', name='Curva ROC XGBoost',
    line=dict(color='darkblue', width=2.5),
    customdata=thresholds,
    hovertemplate='Falsos Positivos (FPR): %{x:.4f}<br>Verdaderos Positivos (TPR): %{y:.4f}<br>Umbral de Corte: %{customdata:.4f}<extra></extra>'
))
fig3.update_layout(
    title='📆 Gráfica 3: Curva Característica Operativa del Receptor (ROC) - Capacidad de Discriminación',
    xaxis_title='Tasa de Falsos Positivos (1 - Especificidad)',
    yaxis_title='Tasa de Verdaderos Positivos (Sensibilidad / Recall)',
    template='plotly_white', hovermode='closest'
)
fig3.show()

# =====================================================================
# 4. SECCIÓN DE EVALUACIÓN DE MÉTRICAS MATEMÁTICAS BANCARIAS (.4f)
# =====================================================================
print("\n🧮 Calculando auditoría de métricas de rendimiento globales del Módulo 3...")
mae_global = mean_absolute_error(y_test, predicciones_prob)
rmse_global = np.sqrt(mean_squared_error(y_test, predicciones_prob))
r2_global = r2_score(y_test, predicciones_prob)
accuracy = accuracy_score(y_test, predicciones_binarias)
auc_roc = roc_auc_score(y_test, predicciones_prob)

print("\n================ METRICAS FINANCIERAS DE RETENCIÓN (.4f) ================")
print(f" Error Absoluto Medio (MAE):               {mae_global:.4f}")
print(f" Raíz del Error Cuadrático (RMSE):         {rmse_global:.4f}")
print(f" Coeficiente de Determinación (R² Score):   {r2_global:.4f}")
print(f" Exactitud Global de Retención (Accuracy): {accuracy:.4f}")
print(f" Área Bajo la Curva ROC (AUC-ROC):         {auc_roc:.4f} (Objetivo de Negocio: > 0.85)")
print("=========================================================================")


🔌 Conectando con Google Drive e importando binarios de Churn...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Modelo final de XGBoost cargado correctamente.

📊 Generando Gráfica 1: Importancia de Variables en Churn...


📊 Generando Gráfica 2: Matriz de Confusión...


📊 Generando Gráfica 3: Curva ROC...



🧮 Calculando auditoría de métricas de rendimiento globales del Módulo 3...

================ METRICAS FINANCIERAS DE RETENCIÓN (.4f) ================
 Error Absoluto Medio (MAE):               0.1266
 Raíz del Error Cuadrático (RMSE):         0.2558
 Coeficiente de Determinación (R² Score):   0.2167
 Exactitud Global de Retención (Accuracy): 0.9120
 Área Bajo la Curva ROC (AUC-ROC):         0.9026 (Objetivo de Negocio: > 0.85)


In [6]:
import os
from google.colab import drive

print("🔌 Conectando con Google Drive para automatizar el README financiero...")
drive.mount('/content/drive')

ruta_drive = '/content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA'
os.makedirs(ruta_drive, exist_ok=True)
ruta_readme = os.path.join(ruta_drive, 'README_FINANCIERO.md')

contenido_readme = """# Ecosistema de Machine Learning Aplicado al Sector Bancario y Financiero 🏦📉

Este repositorio contiene una suite avanzada de modelos analíticos desarrollados bajo estándares de la industria bancaria utilizando **LightGBM**, **XGBoost**, **SHAP** y **Optuna**. El ecosistema aborda dos de las problemáticas de negocio más críticas y de mayor impacto financiero en las instituciones financieras: la gestión del riesgo de impago (*Credit Scoring*) y el control de la deserción de usuarios (*Customer Churn*).

---

## 📁 Arquitectura del Directorio Corporativo (MLOps)

El entorno implementa persistencia binaria estructurada y aislamiento de matrices estadísticas. Todo el ecosistema de producción se sincroniza automáticamente en el siguiente árbol de directorios:

```text
/content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA/
├── datos_credit_scoring_bancario.csv   # Dataset del Módulo 1 (Riesgo Crediticio)
├── datos_churn_bancario.csv            # Dataset del Módulo 3 (Fuga de Clientes)
├── modelo_scoring_bancario.pkl         # Binario y metadatos del Módulo 1 (LightGBM)
└── modelo_churn_bancario.pkl           # Binario y metadatos del Módulo 3 (XGBoost + Optuna)
```

---

## 🏗️ Desglose Metodológico de las Soluciones Bancarias

### 💳 1. Credit Scoring Explicable y Regulado (LightGBM + SHAP)
*   **Caso de Uso de Negocio:** Clasificación analítica del riesgo de impago (*Default*) para solicitudes de crédito corporativas y de consumo, cumpliendo rigurosamente con los marcos regulatorios internacionales (como Basilea III) que exigen explicabilidad jurídica.
*   **Metodología:** Clasificador basado en el algoritmo de gradiente acelerado ligero **LightGBM**. La "caja negra" del modelo es auditada de extremo a extremo mediante valores de Shapley (**SHAP**), permitiendo descomponer linealmente el impacto de características críticas como los ingresos anuales, la edad y el ratio deuda-ingreso de forma individualizada.

### 🏃 2. Predicción de Fuga de Clientes con Optimización Bayesiana (XGBoost + Optuna)
*   **Caso de Uso de Negocio:** Identificación proactiva de patrones transaccionales anómalos o decrementos de actividad que indican una cancelación inminente de cuentas bancarias, permitiendo el despliegue automático de campañas de retención focalizadas.
*   **Metodología:** Clasificador basado en **XGBoost** optimizado de manera avanzada mediante **Optuna** (Optimización Bayesiana). La librería ejecuta búsquedas inteligentes sobre hiperparámetros complejos (`learning_rate`, `max_depth`, `subsample`, `colsample_bytree`) minimizando la métrica de entropía cruzada (`logloss`) en una fracción del tiempo de los enfoques iterativos tradicionales.

---

## 📊 Paneles Operativos e Interactividad Avanzada (Plotly)

Las herramientas de control estadístico han sido desarrolladas con **Plotly (HTML dinámico)** bajo un esquema riguroso de precisión de **4 cifras decimales (`.4f`)** tanto en interfaces corporativas como en cuadros informativos flotantes (*hover*):

1.  **Gráficos de Importancia de Variable Relativa:** Barras horizontales limpias con mapeo numérico porcentual explícito y formateado afuera de cada eje visual.
2.  **Matrices de Confusión de Control Operativo:** Mapas de calor interactivos que permiten auditar los falsos negativos y falsos positivos críticos para balancear las pérdidas de capital del banco contra los costos de campañas de marketing.
3.  **Curvas Características del Receptor (ROC Curve):** Trazado matemático continuo de la sensibilidad contra la especificidad, permitiendo inspeccionar de forma individualizada el umbral de corte óptimo de probabilidad financiera al pasar el ratón por encima de la curva.

---
**Desarrollado bajo estrictos criterios de arquitectura de software y ciencia de datos aplicados al sector de servicios financieros de alta escala.** 🚀
"""

try:
    with open(ruta_readme, 'w', encoding='utf-8') as f:
        f.write(contenido_readme)
    print(f"✅ ¡Archivo README_FINANCIERO.md creado exitosamente en: {ruta_readme}!")
except Exception as e:
    print(f"❌ Error al escribir el archivo: {e}")


🔌 Conectando con Google Drive para automatizar el README financiero...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ ¡Archivo README_FINANCIERO.md creado exitosamente en: /content/drive/MyDrive/COLAB_NOTEBOOKS/MODELOS_ML_PARA_PEDIR_CHAMBA/README_FINANCIERO.md!
